## Notebook content

This notebook lets you load a chosen AutoGluon model from S3, and run predictions. 


 💡 **Tips:**
- Ensure the S3 connection to pipeline run results is configured so the notebook can access run artifacts.
- The model name must match one of the models listed in the leaderboard (the **model** column).

### Contents
This notebook contains the following parts:

**[Setup](#setup)**  
**[Experiment run details](#experiment-run-details)**  
**[Download trained model](#download-trained-model)**  
**[Model insights](#model-insights)**  
**[ROC and precision-recall curves](#roc-and-precision-recall-curves)**  
**[Load the predictor](#load-the-predictor)**  
**[Predict the values](#predict-the-values)**  
**[Summary and next steps](#summary-and-next-steps)**

<a id="setup"></a>
## Setup

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import os

os.environ["PIP_EXTRA_INDEX_URL"] = (
    "https://console.redhat.com/api/pypi/public-rhai/rhoai/3.5/cpu-ubi9-test/simple/"
)
%pip install autogluon.tabular[lightgbm,xgboost,tabm,fastai]==1.5.0+rhaiv.4 | tail -n 1

<a id="experiment-run-details"></a>
## Experiment run details

Set the pipeline name and run ID that identify the training run whose artifacts you want to load. These values are typically available from the pipeline run or workbench.

In [ ]:
pipeline_name = "autogluon-tabular-training-pipeline"
run_id = "9ec21d90-baa0-4a6b-bb2a-40d9d4b43c54"
model_name = "XGBoost_BAG_L1_FULL"

<a id="download-trained-model"></a>
## Download trained model

 📌 **Action:** Ensure the S3 connection is added to the workbench so the notebook can access the results.

Download model binaries and metrics.

In [ ]:
import boto3
import os
import warnings

from botocore.exceptions import SSLError

required_env_vars = (
    "AWS_S3_ENDPOINT",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "AWS_S3_BUCKET",
)
missing_vars = [name for name in required_env_vars if not os.environ.get(name, "").strip()]

if missing_vars:
    raise ValueError(
        f"Missing required environment variable(s): {', '.join(missing_vars)}. "
        "Attach your S3 connection to this notebook workbench and try again."
    )


def _get_s3_bucket(verify=True):
    return boto3.resource(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        verify=verify,
    ).Bucket(os.environ["AWS_S3_BUCKET"])


def _download_model(bucket):
    # The combined models artifact is written by the autogluon-models-training component.
    # model_name already includes the _FULL suffix (e.g. LightGBM_BAG_L1_FULL).
    training_prefix = os.path.join(pipeline_name, run_id, "autogluon-models-training")
    model_subpath = os.path.join("models_artifact", model_name)
    marker = model_subpath + "/"
    base_dir = os.path.abspath(model_name)
    for obj in bucket.objects.filter(Prefix=training_prefix):
        if obj.key.endswith("/") or marker not in obj.key:
            continue
        relative = obj.key.split(marker, 1)[1]
        relative = os.path.normpath(relative)
        if relative == "." or relative.startswith("..") or os.path.isabs(relative):
            warnings.warn(f"Skipping due to invalid path detected: {obj.key!r}")
            continue
        target = os.path.join(model_name, relative)
        abs_target = os.path.abspath(target)
        if abs_target != base_dir and not abs_target.startswith(base_dir + os.sep):
            warnings.warn(f"Skipping due to invalid path detected: {obj.key!r}")
            continue
        parent = os.path.dirname(abs_target)
        if parent:
            os.makedirs(parent, exist_ok=True)
        bucket.download_file(obj.key, target)


best_model_path = model_name  # download into a local directory named after the model
try:
    _download_model(_get_s3_bucket())
except SSLError:
    warnings.warn("SSL error when accessing S3, retrying with verify=False")
    _download_model(_get_s3_bucket(verify=False))

print("Model artifact stored under", best_model_path)

<a id="model-insights"></a>
## Model insights

Display the metrics, confusion matrix, ROC/PR curves, and feature importances for selected model.

### Metrics

Metrics determined on the basis of test data.

In [ ]:
import pandas as pd
import json

with open(os.path.join(best_model_path, "metrics", "metrics.json")) as f:
    metrics = pd.json_normalize(json.load(f))

metrics

### Confusion matrix

In [ ]:
confusion_matrix = pd.read_json(os.path.join(best_model_path, "metrics", "confusion_matrix.json"))
confusion_matrix.head()

<a id="roc-and-precision-recall-curves"></a>
### ROC and precision-recall curves

Precomputed curve points from the pipeline test split (`metrics/curves.json`). This file is produced only for **binary** and **multiclass** classification runs (not regression). **Binary:** ROC and precision-recall plots. **Multiclass:** macro metrics summary (per-class curves are in the JSON under `roc_curve.per_class` / `precision_recall_curve.per_class`).

In [ ]:
import json
import matplotlib.pyplot as plt

curves_path = os.path.join(best_model_path, "metrics", "curves.json")
if not os.path.exists(curves_path):
    print(
        "curves.json was not found for this model. "
        "It is written only for successful binary/multiclass pipeline runs."
    )
else:
    with open(curves_path) as f:
        curves = json.load(f)

    if curves["task_type"] == "binary":
        roc = curves["roc_curve"]
        plt.figure(figsize=(6, 4))
        plt.plot(roc["fpr"], roc["tpr"], label=f"ROC (AUC={roc['auc']:.3f})")
        plt.plot([0, 1], [0, 1], "--", color="gray", label="Random")
        plt.xlabel("False positive rate")
        plt.ylabel("True positive rate")
        plt.title("ROC curve")
        plt.legend()
        plt.show()

        pr = curves["precision_recall_curve"]
        plt.figure(figsize=(6, 4))
        plt.plot(pr["recall"], pr["precision"], label=f"PR (AP={pr['average_precision']:.3f})")
        plt.axhline(pr["baseline_precision"], linestyle="--", color="gray", label="Baseline")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision-recall curve")
        plt.legend()
        plt.show()
    else:
        print(
            f"Multiclass curves ({curves.get('strategy', 'ovr')}); "
            f"macro ROC AUC: {curves['roc_curve']['auc_macro']:.3f}; "
            f"macro AP: {curves['precision_recall_curve']['average_precision_macro']:.3f}"
        )

### Feature importance
Top ten are displayed.

In [ ]:
feature_importance = pd.read_json(os.path.join(best_model_path, "metrics", "feature_importance.json"))
feature_importance.head(10)

<a id="load-the-predictor"></a>
## Load the predictor

Load the trained model as a TabularPredictor object.

In [ ]:
from autogluon.tabular import TabularPredictor

predictor = TabularPredictor.load(os.path.join(best_model_path, "predictor"), require_version_match=False)

<a id="predict-the-values"></a>
## Predict the values

Use sample records to predict values. 

In [ ]:
import pandas as pd

score_data = [{'PassengerId': 590, 'Pclass': 3, 'Name': 'Murdlin, Mr. Joseph', 'Sex': 0, 'Age': None, 'SibSp': 0, 'Parch': 0, 'Ticket': 'A./5. 3235', 'Fare': 8.05, 'Cabin': None, 'Embarked': 'S'}]
score_df = pd.DataFrame(data=score_data)
score_df.head()

Predict the values using `predict_proba` method.

In [ ]:
predictor.predict_proba(score_df)

<a id="summary-and-next-steps"></a>
## Summary and next steps

**Summary:** This notebook loaded a trained AutoGluon model from S3 and ran predictions on sample data using `predict_proba`.

**Next steps:**
- Run predictions on your own data (ensure columns match the training schema).
- Try another model from the leaderboard by changing `model_name` and re-running the download and load cells.
- Optionally create the Predictor online deployment using KServe custom runtime.

---